# Mapping Excel to SPOD
- Prerequisites: 
  - Anaconda packages: `pandas, openpyxl`

This script imports the Mapping Excel sheet and processes it into 
SPOD JSON format, which can be imported back. 

## Result

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet

## Configuration
The following parameters has to be definded when running as regular python script

In [ ]:
MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
MAPPING = 'testdata/mapping_sika-2022-04-21.xlsx'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import shutil
import copy
from pathlib import Path

In [ ]:
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
# openpyxl
import openpyxl
from openpyxl.worksheet.table import Table
from openpyxl.utils import cell
from openpyxl.styles import PatternFill

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
import jsonpath_ng as jsonpath

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

In [ ]:
spod_file = Path(MODEL)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
logger.info(f"Loaded SPOD {spod['model']['name']} from '{spod_file.resolve()}' revision {spod['_imprint_'].get('git-revision','???')}")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Use the fyayc SPOD library

### Tools path

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

In [ ]:
from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

# Changes in mapping.xlsx versus previous version
Compare new mapping.xlsx with previous version and update the SPOD accordingly.
Create a log of changes in JSON format, close to the SPOD structure.

In [ ]:
wb = openpyxl.load_workbook(MAPPING)
assert wb['Mapping'] is not None, f"No sheet named 'mapping' in workbook '{OLD_MAPPING}'"
print(f"Loaded workbook {MAPPING}")

# Compare mapping sheet against SPOD 

In [ ]:
mappings = wb['Mapping']
print(f"Sheet {mappings} contains {mappings.tables.keys()}")
mapping_table = mappings.tables['mapping']

print(f"Processing table '{mapping_table.displayName}' spanning {mapping_table.ref} with headers {mapping_table.headerRowCount}")
table_range_tuple = cell.range_boundaries(mapping_table.ref)

first_data_row = table_range_tuple[1] + mapping_table.headerRowCount
headers = mappings[table_range_tuple[1]]

In [ ]:
column_mapping = dict()

index = 0
for column in headers:
    column_mapping[column.value] = index
    index += 1

list(column_mapping.items())[:18], len(column_mapping)

## jspath templates to access attributes

In [ ]:
columns = { 'Examples': '$.attributes["{attr}"].examples["en"]' }

for skey, system in spod['systems'].items():
    columns[system['name']] = '$.columns["{col}"]["name"]'

In [ ]:
columns

In [ ]:
def build_json_path(path: str, **kwargs) -> jsonpath.Child:
    expanded = path.format(**kwargs)
    path = jsonpath.parse(expanded)
    return path

In [ ]:
path = build_json_path(columns['Examples'], attr='ATTR249')
path

In [ ]:
path.find(spod)[0].value

In [ ]:
def process_row(headers: tuple, row: tuple):
    
    attrid = row[column_mapping['AID']]
    
    for header in headers:
        index = header.col_idx - 1
        new_value = row[index]
        
        colmapping = columns.get(header.value)
        if colmapping is not None:
            jpath = build_json_path(colmapping, attr=attrid)
            hits = jpath.find(spod)
            
            # accept empty
            if len(hits) > 0 and new_value is not None:
                assert len(hits) == 1, f"Expecting only one precise hit, got: {hits}"
                current_value = hits[0].value
                if current_value != new_value:
                    logging.info(f"Updating current value '{current_value}' to '{new_value}'")
                    jpath.update(spod, new_value)
            else:
                logging.debug(f"Cell and new value are undefined")
        else:
            print(f"Column {header.value} = {new_value}")
        


In [ ]:
sample_row = next(mappings.iter_rows(min_row=first_data_row, max_row=first_data_row + 1, values_only=True))
process_row(headers, sample_row)

In [ ]:
create_statement = f"Created by import"
user = 'bue'
stamp = '2022-04-28 15:38:01 UTC'

In [ ]:
def update_column(column, name: str, tech: str, row: tuple) -> []:
    updated = copy.deepcopy(column)
    updated['um'] = user
    updated['dm'] = stamp
    return updated

In [ ]:
new_table_cache = dict()

def negative_number_generator() -> int:
    counter = 0
    while True:
        counter -= 1
        yield counter
    
new_element_id_generator = negative_number_generator()

def process_system_row(spod: dict, system_key: str, key, name, tech, row):
    column_key = None
    column = None
    
    result = []
    
    if key is not None:
        index = key.rfind(':')
        colkey = key[index+1:]
        column = spod['columns'].get(colkey)
    
    column_by_name = None
    if column is None and name is not None and len(name) > 0:
        hits = list(filter(lambda c: c['interface-id+'] == system_key and c['name'] == name, spod['columns'].values()))
        if len(hits) == 1:
            column_by_name = hits[0]
        else:
            assert len(hits) == 0, f"Found several matching columns for name {name} in interface {key}! {hits}"
    
    if column is not None and column_by_name is not None and column_by_name != column:
        logging.error(f"Column reference and name error! {column} vs {column_by_name}")
        
    attributes = []
    if column is not None:
        logging.debug(f"Found column reference for update: {key}")
        updated = update_column(column, name, tech, row)
        if updated is not None:
            result.append( ('u', colkey, updated) )
    else:
        if name is not None or tech is not None:
            logging.info(f"Creating new column '{name}' for system '{system_key}")
            if name is None:
                name = tech
            tables = spod['systems'][system_key]['tables+']
            
            if len(tables) > 0:
                table_id = tables[0]
            else:
                table_id = 'TABL' + str(next(new_element_id_generator))
                new_table = new_table_cache.get(table_id)
                if new_table is None:
                    new_table = { 'name': 'main', 'interface-id': system_key, 'prefix': None, 'descr': create_statement, 
                                 'uc': user, 'dc': stamp, 'um': None, 'dm': None
                                }
                    new_table_cache[table_id] = new_table
                    result.append( ('c', table_id, new_table) )
                
            result.append( ('c', f'COLU-{system_key}-{row[0].row}', { 'name': name, 'table-id': table_id, 'interface_col_id': tech,
                    'uc': user, 'dc': stamp, 'attributesmapped': attributes,
                    'mandatory': False, 'datatype': 'unknown', 'format': None, 'domain': None, 'descr': create_statement,
                    }) )
    
    for _, _, element in result: 
        ref = element.get('sourceref')
        if ref is None:
            element['sourceref'] = { 'excel-mapping-import': [ MAPPING ] }
        else:
            ref['excel-mapping-import'] = { 'excel-mapping-import': [ MAPPING ] }
    
    return result

In [ ]:
all_changes = []
for skey, system in spod['systems'].items():
    
    key_col_index = column_mapping[skey + ':FQN']
    name_col_index = key_col_index + 1
    tech_col_index = key_col_index + 2
    
    data_iterator = mappings.iter_rows(min_row=first_data_row, values_only=False)
    changes = []
    for row in tqdm(data_iterator, desc=f"Processing rows of system {system['name']}", unit=" Row", dynamic_ncols=True):
        value = row[name_col_index].value
        elements = process_system_row(spod, skey, key=row[key_col_index].value, name=row[name_col_index].value, tech=row[tech_col_index].value, row=row)
        changes += elements
    if len(changes) > 0:
        logging.info(f"Applying {len(changes)} element changes for system {system['name']}")
        all_changes += changes

In [ ]:
len(all_changes)

In [ ]:
spod_updated = copy.deepcopy(spod)
for action, key, element in all_changes:
    print(f"Processing {action} on {key}")
    if action == 'c':
        if key.startswith('COLU'):
            logging.info(f"Creating column {key}")
            spod_updated['columns'][key] = element
        if key.startswith('TABL'):
            logging.info(f"Creating table {key}")
            spod_updated['tables'][key] = element
    if action == 'u':
        if key.startswith('COLU'):
            logging.info(f"Updating column {key}")
            spod_updated['columns'][key] = element

In [ ]:
updated = Path(MODEL).with_suffix('.new.json')
with open(updated, 'w') as out:
    json.dump(spod_updated, out, indent=4)
print(f"Wrote {updated}")

# Try to merge

In [ ]:
from LOAD_MODELS.LOAD_INFRA import mergedbs
from SSOT_db.SQL_INFRA import dbConnect
from SSOT_db.IM_JSON.jsbase import JSModel

In [ ]:
sourcedb = Path(MODEL).with_suffix('.db')
assert sourcedb.is_file()

tempdb = sourcedb.with_suffix('.new.db')
# clean slate
tempdb.unlink(missing_ok=True)

shutil.copy(sourcedb, tempdb)
assert tempdb.is_file()

In [ ]:
conn = dbConnect.openDB(pfilepath=str(tempdb))

In [ ]:
print(f"Merging updated json into existing DB")

new_model = JSModel(spod_updated)
mergedbs.mergejson2db(pmodeljson=new_model)

## List structure definition

In [ ]:
def emit_row(spod: dict, attribute_key: str, attribute: dict, translator: Translator) -> []:
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"
    result = [
        enti_key + ':' + attribute_key,
        enti_key,
        attribute_key,
        translator.tr(entity['name'], 'en'),
        translator.tr(attribute['name'], 'en'),
        translator.tr(entity['name'], 'de'),
        translator.tr(attribute['name'], 'de'),
        translator.tr(entity['name'], 'fr'),
        translator.tr(attribute['name'], 'fr'),
        ', '.join(translator.tr(attribute.get('examples'), 'en')),
        translator.tr(attribute['descr'], 'en'),
        len(attribute['columnsmapped+']),
    ]

    for skey in spod['systems'].keys():
        mapping = firsthit(spod, attribute_key, skey)
        result = result + mapping

    return result

In [ ]:
headings = headings_im + systems
f"Columns ({len(headings)}): {', '.join(headings)}"

# Create data table (content)

In [ ]:
data_table = [ emit_row(spod, key, attribute, translator) for key, attribute in spod['attributes'].items() ]

In [ ]:
len(data_table)

In [ ]:
f"Already mapped {len(columns_mapped)} columns of {len(spod['columns'])}"

In [ ]:
list(columns_mapped.items())[0:3]

## Append unmapped columns to the bottom

In [ ]:
def aux_row(spod: dict, key: str, column: dict, translator: Translator) -> []:
    mapped = column['attributesmapped']
    if len(mapped) > 0:
        attrkey = mapped[0]
        attr = spod['attributes'][attrkey]
        result = emit_row(spod, attrkey, attr, translator)
        result.extend( [ None ] * (len(headings) - len(result))  )
    else:
        result = [ None ] * len(headings)
    
    map_count = len(column['attributesmapped'])
    result[len(headings_im) - 1] = map_count

    index = system_index[column['interface-id+']]
    values = export_column(key, column)
    result[index + 0] = values[0]
    result[index + 1] = values[1]
    result[index + 2] = values[2]
    
    
    return result

In [ ]:
remainder = [ aux_row(spod, key, spod['columns'][key], translator) for key in filter(lambda key: key not in columns_mapped.keys(), spod['columns'].keys()) ]

In [ ]:
data_table = data_table + remainder

In [ ]:
### Sort by Attribute FQN
data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')

# Prepare Excel Workbook

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = 'Mapping'

# add column headings. NB. these must be strings
ws.append(headings)
for row in data_table:
    ws.append(row)

## Define a data table readable by Sharepoint

In [ ]:
tab = Table(displayName="Mapping", ref=f"A1:{get_column_letter(len(headings))}{len(data_table)+1}")
ws.add_table(tab)
tab._initialise_columns()

for column, value in zip(tab.tableColumns, headings):
    column.name = value

## Styling

In [ ]:
pal = list(sns.color_palette('pastel'))

for column_index in range(len(headings_im), len(headings)):
    color_index = int((column_index - len(headings_im)) / 3)
    color = pal[color_index % len(pal)]
    rgb = str(mcolors.to_hex(color))[1:]
    #print(rgb)
    for cell in ws[get_column_letter(column_index + 1)]:
        cell.fill = PatternFill(fgColor=rgb, fill_type = "solid")

### Resize and hide columns

In [ ]:
ws.column_dimensions['B'].hidden= True
ws.column_dimensions['C'].hidden= True

ws.column_dimensions['F'].hidden= True
ws.column_dimensions['G'].hidden= True
ws.column_dimensions['H'].hidden= True
ws.column_dimensions['I'].hidden= True


In [ ]:
base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + 1 + (index * 3)
    col_letter = get_column_letter(colnr)
    dim = ws.column_dimensions[col_letter]
    dim.hidden= True
    
    col_letter_cid = get_column_letter(colnr + 2)
    dim = ws.column_dimensions[col_letter_cid]
    dim.hidden= True
    
    print(f"Hiding columns {col_letter} ({ws[col_letter + '1'].value})"
          f" and {col_letter_cid} ({ws[col_letter_cid + '1'].value}) on System {system['name']} idx {colnr}")
    
    index += 1

## Save to Excel file

In [ ]:
dest = Path(DESTINATION)
wb.save(dest)
print(f"Wrote {dest.resolve()}")

# xlsxwriter Approach

In [ ]:
import xlsxwriter

In [ ]:
xlsx_destination = 'IM_' + dest.name
workbook = xlsxwriter.Workbook(xlsx_destination)

title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})



In [ ]:
## Summary is first sheet, but will be filled last
summary = workbook.add_worksheet('Summary')

In [ ]:
## Prepare Mapping sheet

In [ ]:
worksheet = workbook.add_worksheet('Mapping')

col = 0
for header in headings:
    worksheet.write(0, col, header)
    col += 1

row = 1
for entry in data_table:
    col = 0
    for item in entry:
        worksheet.write(row, col, item)
        col += 1
    row += 1

### Define Table

In [ ]:
table_column_headers = [ { 'header': name } for name in headings ]

In [ ]:
worksheet.add_table(0, 0, len(data_table) + 1, len(headings) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

### Styling

In [ ]:
# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + (index * 3)
    worksheet.set_column(colnr, colnr, None, None, { 'hidden': 1, })
    
    # Column name on system
    worksheet.set_column(colnr + 1, colnr + 1, 30)
    
    # Technical reference
    worksheet.set_column(colnr + 2, colnr + 2, None, None, { 'hidden': 1, })        
    index += 1

## Add one sheet per system

In [ ]:
def fill_worksheet(spod: dict, skey: str, system: str, sheet):
    row = 0
    sheet.write(row, 0, 'Table Key', column_head_format)
    sheet.write(row, 1, 'Table Name', column_head_format)
    sheet.write(row, 2, 'Column Key', column_head_format)
    sheet.write(row, 3, 'Tech-ID', column_head_format)
    sheet.write(row, 4, 'Name', column_head_format)
    sheet.write(row, 5, 'Description', column_head_format)
    sheet.write(row, 6, '|', column_head_format)
    sheet.write(row, 7, 'IM Attributes', column_head_format)
    
    row += 1
    columns = sorted(list(spod['columns'].items()), key=lambda c: c[1]['table-name+'])
    for ckey, column in columns:
        if column['interface-id+'] == skey:
            sheet.write(row, 0, column['table-id'])
            sheet.write(row, 1, column['table-name+'])
            sheet.write(row, 2, ckey)
            sheet.write(row, 3, column['interface_col_id'])
            sheet.write(row, 4, column['name'])
            sheet.write(row, 5, translator.tr(column['descr'], 'de'))
            sheet.write(row, 6, '|')
            sheet.write(row, 7, ', '.join(column['attributesmapped']))
            row += 1
    
    return row

In [ ]:
import re

worksheets = dict()

for key, system in spod['systems'].items():
    title = re.sub(r'[\:\[\]*?/\\]', '_', system['name'])
    length = min(25, len(title))
    t = key.replace('INTF','') + ' ' + title[:length]
    if t.lower() in worksheets.keys():
        t = key.replace('INTF','') + ' ' + title[max(0, len(title) - 25):]
    worksheet = workbook.add_worksheet(t)
    worksheets[t.lower()] = worksheet
    rows = fill_worksheet(spod, key, system, worksheet)
    print(f"{key}: {system['name']} -> {t} with {rows} rows")

## Summary sheet

In [ ]:
summary.write(0, 0, "Summary", title_format)

row = 2
summary.write(row, 0, 'Key', column_head_format)
summary.write(row, 1, 'Name', column_head_format)
summary.write(row, 2, 'Mapped', column_head_format)
summary.write(row, 3, 'Total', column_head_format)
summary.write(row, 4, 'Tables', column_head_format)
summary.write(row, 5, 'Sheet Link', column_head_format)

row = 3
for skey, system in spod['systems'].items():
    summary.write(row, 0, skey)
    summary.write(row, 1, system['name'])
    
    columns = list(filter(lambda c: c['interface-id+'] == skey, spod['columns'].values()))
    mapped = list(filter(lambda c: len(c['attributesmapped']) > 0, columns))
    summary.write(row, 2, len(mapped))
    summary.write(row, 3, len(columns))
    
    summary.write(row, 4, len(system['tables+']))
    
    _, ws = next(iter(filter(lambda t: skey[4:] in t[0], worksheets.items())))
    summary.write_url(row, 5, f"internal:'{ws.get_name()}'!A1", string=f'Sheet {skey[4:]}')
    
    row += 1

## Styling

In [ ]:
summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

## Write Excel file

In [ ]:
workbook.close()
print(f"Wrote {xlsx_destination}")

# Visually verify

In [ ]:
import subprocess
r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)